In [1]:
import json
import os
from pathlib import Path
from typing import Optional

import pandas as pd
import numpy as np
import torch

from logging_setup import create_logger
from phi_3_5_constants import hidden_state_size, seed, dsets_folder, token_lengths_path, train_split_records_path, \
    validation_split_records_path
from phi_3_5_probe import learn_directions_and_train_probe, PolarityAwareTruthProbe

In [2]:
np_rng = np.random.default_rng(seed)
torch_rng = torch.Generator().manual_seed(seed)
torch.manual_seed(seed)

In [3]:
logger = create_logger(__name__)

In [4]:
activations_data_folder= Path("D:\\TruthIsUniversal_In_Phi_3_5_Mini\\best2_layers_activations_for_final_token_of_sequences")
results_folder = Path("learned_vectors_n_probes")

In [5]:
results_folder.mkdir(exist_ok=True)

In [6]:
dsets_index_df = pd.read_csv(dsets_folder / "datasets_index.csv", index_col="Idx")

In [7]:
num_dsets = dsets_index_df.shape[0]

In [8]:
with token_lengths_path.open("r") as f:
    record_lengths_in_tokens = json.load(f)
with train_split_records_path.open("r") as f:
    train_split_record_idxs = json.load(f)
with validation_split_records_path.open("r") as f:
    validation_split_record_idxs = json.load(f)

In [9]:
all_dsets_activations: list[torch.Tensor] = [torch.zeros(1) for _ in range(num_dsets)]
all_dsets_labels: list[torch.Tensor] = [torch.zeros(1) for _ in range(num_dsets)]

In [10]:
train_data_lyr18_avg_activs_by_dset = torch.zeros((num_dsets, hidden_state_size))
train_data_lyr25_avg_activs_by_dset = torch.zeros((num_dsets, hidden_state_size))

#0th dimension is indexed by dataset id; "train_data" means only based on the train segment of the dataset
train_data_lyr18_truth_dirs: torch.Tensor = torch.zeros((num_dsets, hidden_state_size))
train_data_lyr18_polarity_dirs: torch.Tensor = torch.zeros((num_dsets, hidden_state_size))
# TODO do the same for layer25 activations (and later also layer18+layer25 concatenated activations) after confirming that the learned truth/polarity directions for layer 18 on a given dataset can effectively be used by a linear classifier to predict truth/falsehood in validation segment of same dataset
train_data_lyr25_truth_dirs: torch.Tensor = torch.zeros((num_dsets, hidden_state_size))
train_data_lyr25_polarity_dirs: torch.Tensor = torch.zeros((num_dsets, hidden_state_size))

lyr18_probes: list[Optional[PolarityAwareTruthProbe]] = [None for _ in range(num_dsets)]
lyr25_probes: list[Optional[PolarityAwareTruthProbe]] = [[None for _ in range(num_dsets)]]

In [11]:
num_datasets_finished_so_far = 0

In [ ]:
for dset_idx, dset_dtls in dsets_index_df.iterrows():
    if dset_idx < num_datasets_finished_so_far:
        continue#i.e. the single-layer/single-dataset directions/probes for this dataset have already been learned
    categ_nm = dset_dtls["Categ_Folder"]
    dset_file_nm = dset_dtls["Dataset_File"]
    dset_nm = os.path.splitext(dset_file_nm)[0]
    
    dataset = pd.read_csv(dsets_folder / categ_nm / dset_file_nm)
    dset_size = dataset.shape[0]
    dset_labels = torch.from_numpy(dataset['label'].to_numpy().astype(np.float32)[:, np.newaxis])
    all_dsets_labels[dset_idx] = dset_labels
    
    activs_path = activations_data_folder / categ_nm / (dset_nm + ".pt")
    relevant_activations = torch.load(activs_path, weights_only=True)
    assert relevant_activations.shape == (2, dset_size, hidden_state_size)
    all_dsets_activations[dset_idx] = relevant_activations
    
    train_split_activs = relevant_activations[:, train_split_record_idxs[str(dset_idx)], :]
    train_split_truth_labels = dset_labels[train_split_record_idxs[str(dset_idx)], :]
    
    train_split_polarity_labels = torch.ones(train_split_truth_labels.shape)
    if dset_dtls["is_negated"]:
        train_split_polarity_labels *= -1
    
    val_split_activs = relevant_activations[:, validation_split_record_idxs[str(dset_idx)], :]
    val_split_truth_labels = dset_labels[validation_split_record_idxs[str(dset_idx)], :]
    
    categ_output_folder = results_folder / categ_nm
    categ_output_folder.mkdir(exist_ok=True)
    
    logger.debug(f"doing direction-learning and probe training for layer 18 of dataset {dset_nm} in category {categ_nm}")
    lyr18_probe = learn_directions_and_train_probe(
        train_split_activs[0,:,:], train_split_truth_labels, train_split_polarity_labels, val_split_activs[0,:,:], val_split_truth_labels, np_rng)
    
    train_data_lyr18_truth_dirs[dset_idx, :] = lyr18_probe.truth_dir.clone().cpu().T
    train_data_lyr18_polarity_dirs[dset_idx, :] = lyr18_probe.polarity_dir.clone().cpu().T
    train_data_lyr18_avg_activs_by_dset[dset_idx, :] = lyr18_probe.mean_activation.clone().cpu().T
    lyr18_probes[dset_idx] = lyr18_probe
    
    torch.save(lyr18_probe.state_dict(), categ_output_folder / f"{dset_nm}_lyr18.pth")
    
    logger.debug(f"doing direction-learning and probe training for layer 25 of dataset {dset_nm} in category {categ_nm}")
    lyr25_probe = learn_directions_and_train_probe(
        train_split_activs[1,:,:], train_split_truth_labels, train_split_polarity_labels, val_split_activs[1,:,:], val_split_truth_labels, np_rng)
    
    train_data_lyr25_truth_dirs[dset_idx, :] = lyr25_probe.truth_dir.clone().cpu().T
    train_data_lyr25_polarity_dirs[dset_idx, :] = lyr25_probe.polarity_dir.clone().cpu().T
    train_data_lyr25_avg_activs_by_dset[dset_idx, :] = lyr25_probe.mean_activation.detach().cpu().T
    lyr25_probes[dset_idx] = lyr25_probe
    
    torch.save(lyr25_probe.state_dict(), categ_output_folder / f"{dset_nm}_lyr25.pth")

2024-10-20 22:43:34,384;phi_3_5_probe;INFO:   Iteration     Total nfev        Cost      Cost reduction    Step norm     Optimality   
2024-10-20 22:43:34,384;phi_3_5_probe;INFO:       0              1         4.1335e+11                                    1.32e+09    


In [19]:
# for dset_idx, dset_dtls in dsets_index_df.iterrows():
#     if dset_idx > 1:
#         break
#     categ_nm = dset_dtls["Categ_Folder"]
#     dset_file_nm = dset_dtls["Dataset_File"]
#     dset_nm = os.path.splitext(dset_file_nm)[0]
#     
#     


In [87]:
# all_dsets_activations[3].shape

torch.Size([2, 500, 3072])

In [10]:
#TODO train on positive + negated dataset pairs (within a category) to confirm whether polarity direction is doing any good 

np.int64(13)

In [68]:
# dummy_truth_probe_animals_pos_lyr_18 = PolarityAwareTruthProbe(torch.ones(hidden_state_size, 1), torch.ones(hidden_state_size,1), torch.ones(hidden_state_size,1))
#NOTE FOR LATER- loading model state dict worked if the probe's buffers had been initialized with all-ones tensors but didn't work in practice if they'd been initialized with all-zeros tensors (everything would look right when examining the loaded probe but any predictions it made would be all nan)
# That is, the buffers of the probe object which was created solely so that its load_state_dict() method could be called

In [70]:
# dummy_truth_probe_animals_pos_lyr_18.load_state_dict(torch.load(results_folder / "animal_class" / "animal_class_lyr18.pth"))

C:\Users\ssili\AppData\Local\Temp\ipykernel_55864\4070788335.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dummy_truth_probe_animals_pos_lyr_18.load_state_dict(torch.l

<All keys matched successfully>

In [71]:
# dummy_truth_probe_animals_pos_lyr_18(all_dsets_activations[0][0, 0:2, :])

tensor([[9.9996e-01],
        [2.6754e-05]], grad_fn=<SigmoidBackward0>)

In [88]:
# activations_ = all_dsets_activations[2][0]
# print(f"activs type={type(activations_)}; shape={activations_.shape}, dtype={activations_.dtype}")
# print(activations_.mean(dim=0))
# preds = dummy_truth_probe_animals_pos_lyr_18(activations_)
# print(f"preds type={type(preds)}; shape={preds.shape}, dtype={preds.dtype}")
# #print(preds)
# labels = torch.from_numpy(all_dsets_labels[2][:, np.newaxis])
# print(f"labels type={type(labels)}; shape={labels.shape}")
# errs = preds - labels
# print(f"errs type={type(errs)}; shape={errs.shape}")
# (MSE := torch.mean(torch.square(errs)).item())

activs type=<class 'torch.Tensor'>; shape=torch.Size([500, 3072]), dtype=torch.float32
tensor([ 0.4619, -0.0388,  0.2940,  ...,  0.6617,  0.0588, -0.9982])
preds type=<class 'torch.Tensor'>; shape=torch.Size([500, 1]), dtype=torch.float32
labels type=<class 'torch.Tensor'>; shape=torch.Size([500, 1])
errs type=<class 'torch.Tensor'>; shape=torch.Size([500, 1])


0.34377041459083557

In [ ]:
# TODO confirm that the learned truth/polarity directions for layer 18 on a given dataset can effectively be used by a linear classifier to predict truth/falsehood in other datasets in group